# 02 - Preprocesamiento de Datos

Pipeline de transformacion para modelado

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('../data/raw/telco_customer_churn.csv')
print(f'Shape original: {df.shape}')
df.head()

In [ ]:
# Eliminar customerID
df = df.drop('customerID', axis=1)

# TotalCharges puede tener espacios vacios
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f'Valores nulos en TotalCharges: {df["TotalCharges"].isnull().sum()}')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

In [ ]:
# Codificar variable objetivo
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Separar variables categoricas y numericas
cat_cols = df.select_dtypes(include='object').columns.tolist()
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols.remove('Churn')

print(f'Variables categoricas: {len(cat_cols)}')
print(f'Variables numericas: {len(num_cols)}')

In [ ]:
# One-Hot Encoding para variables categoricas
df_processed = pd.get_dummies(df, columns=cat_cols, drop_first=True)
print(f'Shape final: {df_processed.shape}')
df_processed.head()

In [ ]:
# Separar features y target
X = df_processed.drop('Churn', axis=1)
y = df_processed['Churn']

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Churn rate train: {y_train.mean():.3f}')
print(f'Churn rate test: {y_test.mean():.3f}')

In [ ]:
# Escalar variables numericas
scaler = StandardScaler()
num_cols_to_scale = [c for c in num_cols if c in X_train.columns]
X_train[num_cols_to_scale] = scaler.fit_transform(X_train[num_cols_to_scale])
X_test[num_cols_to_scale] = scaler.transform(X_test[num_cols_to_scale])

print('Preprocesamiento completado.')
print(f'Features finales: {X_train.shape[1]}')

In [ ]:
# Guardar datos procesados
import os
os.makedirs('../data/processed', exist_ok=True)

train_data = pd.concat([X_train, y_train], axis=1)
test_data = pd.concat([X_test, y_test], axis=1)

train_data.to_csv('../data/processed/train.csv', index=False)
test_data.to_csv('../data/processed/test.csv', index=False)
print('Datos guardados en data/processed/')